<a href="https://colab.research.google.com/github/mkolennikova/TEB-Ru/blob/main/run_in_collab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TEB-Ru Collab Sandbox

This notebook provides a step-by-step pipeline to download, compile, and run the TEB-Ru model for a user-defined location using ERA5 forcing.

Step-by-step pipeline:
- [Model download](#download-model-from-repository)
- [Model compilation](#model-compilation)
- [Forcing download](#download-atmospheric-forcing)
- [Configure & run simulations](#configure-and-run-simulations)

## Set up environment

This cell imports required Python libraries and sets up the working directory. It also defines helper functions for module reloading.

The following dictionary `opts` controls the runtime behaviour:

- `use_gdrive`: if `True`, the notebook will mount Google Drive and use `/content/drive/MyDrive/TEB_work` as the working directory.  
  If `False` (default), it uses the local Colab ephemeral storage at `/content/TEB_work`.

- `reload_repo`: if `True`, the repository will be freshly cloned (overwriting any local changes).  
  If `False` (default), it will only run `git pull` to fetch updates.

- `repo_path`: the URL of the Git repository to clone. You can change this to a fork or a different branch.

**Note:** before running the next cells, you can adjust these options according to your preferences.

In [ ]:
opts = {'use_gdrive': False,
        'reload_repo': False,
        'repo_path': 'https://github.com/mkolennikova/TEB-Ru'}


In [ ]:
import os, sys, shutil, glob, time
import subprocess
import importlib
import pandas as pd
from pathlib import Path

def import_and_reload (module_name: str):
    """Loads a module by name, or reloads it if already imported."""
    if module_name in sys.modules:
        print(f"Reloading existing module: {module_name}")
        return importlib.reload(sys.modules[module_name])
    else:
        print(f"Loading new module: {module_name}")
        return importlib.import_module(module_name)


In [ ]:
if opts['use_gdrive']:
  from google.colab import drive
  drive.mount('/content/drive')
  WORK_DIR = '/content/drive/MyDrive/TEB_work'
else:
  WORK_DIR = '/content/TEB_work'

MODEL_NAME = os.path.basename(os.path.normpath(opts['repo_path']))
MODEL_DIR = WORK_DIR + '/' + MODEL_NAME

os.makedirs(WORK_DIR, exist_ok = True)


## Download model from repository

The next code cell clones the repository or pulls updates if it already exists.

`reload_repo = True`: clone model repository from remote, clearing all local changes  
`reload_repo = False`: update repository by calling `git pull`

In [ ]:
os.chdir (WORK_DIR)
if not os.path.isdir (MODEL_NAME) or opts['reload_repo']:
  if os.path.isdir (MODEL_NAME):
    %rm -r {MODEL_NAME}
  !git clone {opts['repo_path']}
else:
  os.chdir(MODEL_DIR)
  !git pull



### Install necessary python libraries

In [ ]:
sys.path.append (MODEL_DIR + '/python/')
install_utils = import_and_reload ('install_utils')

install_utils.import_extrernal_modules()


## Model compilation

The next cell compiles the Fortran model using the provided Makefile. It logs the output and checks for the generated executable.

In [ ]:
make_commands = ['all']
#make_commands = ['clean', 'all']

run_utils = import_and_reload ('run_utils')

start_time = time.time()

for comand in make_commands:
  run_utils.run_in_cell(exe_path='make',
                        args=['-C', MODEL_DIR, comand],
                        num_lines = 10,
                        log_file=f'{MODEL_DIR}/make_{comand}_output.txt')

new_files = [f for f in glob.glob(MODEL_DIR + '/*.*') if os.path.getmtime(f) >= start_time]

print('\nCompilation finished, new files created:\n' + '\n'.join('\t' + f for f in new_files))

exe_files = [f for f in new_files if f.endswith('.exe')]
if exe_files:
  if len (exe_files) == 1:
    EXE_PATH = exe_files[0]
    EXE_FILE = os.path.basename(EXE_PATH)
    print (f'✅ Compilation is successfull, executable is {EXE_PATH}')
  else:
    print ('❌ Something is wrong: there is more than one new executable file')
else:
    print ('❌ Something is wrong: there are not any new executable file')


### Test model run

This cell runs a quick test simulation with default settings to verify the compilation was successful. Default namelists and forcing are used.

In [ ]:
run_utils = import_and_reload ('run_utils')
run_utils.run_in_cell(exe_path=f'./{EXE_FILE}', cwd = MODEL_DIR, log_file='TEB_log.txt', num_lines=5)

## Download atmospheric forcing

The following cells download ERA5 reanalysis data for the specified location and time range, and convert them into the format required by the model (ASCII forcing files).

When running a first time, provide a key for your CDS API.


In [ ]:
install_utils.init_CDS()

In [ ]:
forcing_ERA5 = import_and_reload ('forcing_ERA5')
os.chdir(WORK_DIR)

forcing_params = dict(lat = 55.030204,
                      lon = 82.920430,
                      base_dir = 'Novosibirsk',
                      start_date = '2024-01-01',
                      end_date = '2024-12-31',
                      time = '1h')


forcing_ERA5.prepare_forcing (**forcing_params)

## Configure and run simulations

### Configure your custom simulation based on downloaded forcing

The code cell below reads the default parameter namelist, applies modifications (e.g., urban canopy parameters or model options), and writes a new namelist for your experiment. It also configures the forcing namelist path and output directory.

In [ ]:
import f90nml

site_name = 'Novosibirsk'
exp_name = 'CTRL'

FORCING_NML_PATH = f'{WORK_DIR}/{site_name}/forcing_ERA5/namelist_forcing.nml'
PARAMS_NML_PATH  = f'{WORK_DIR}/{site_name}/params_{exp_name}.nml'
OUTPUT_DIR       = f'{WORK_DIR}/{site_name}/output_{exp_name}/'

os.makedirs(OUTPUT_DIR, exist_ok = True)

params_nml = f90nml.read(f'{MODEL_DIR}/namelist/namelist.nml')
params_nml['tebparam']['teb_utc_hour'] = 7

# ***** Insert other namelist modifications here ****
# params_nml['tebparam']['...'] = ...

params_nml.write(PARAMS_NML_PATH, force=True)




### Run model for your custom simulation using comand-line arguments

```
# Custom forcing namelist

./TEB_offline.exe -forcing_nml my_forcing.nml

# Custom parameters namelist
./TEB_offline.exe -forcing_nml my_forcing.nml -param_nml my_params.nml -output my_results/

# Custom output folder
./TEB_offline.exe -output my_results/

# Help
./TEB_offline.exe -help

```

In [ ]:
run_utils = import_and_reload ('run_utils')
run_utils.run_in_cell(exe_path=f'./{EXE_FILE}',
                      args = ['-forcing_nml', FORCING_NML_PATH,
                              '-param_nml', PARAMS_NML_PATH,
                              '-output', OUTPUT_DIR],
                      log_file=f'{OUTPUT_DIR}/model_output.log', cwd = MODEL_DIR, num_lines=5)

In [ ]:
import output_utils
output_utils = import_and_reload ('output_utils')

output_df = output_utils.read_output (OUTPUT_DIR, FORCING_NML_PATH)


### Preview simulation results

After the simulation, these cells read the output files and generate interactive (Plotly) and static (Matplotlib) plots for quick visual inspection.

In [ ]:
output_utils.preview_output_mpl (output_df)

In [ ]:
output_utils.preview_output_plotly (output_df)